Import the libraries required for Differential Gene Expression Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

Load Data

In [ ]:
# Load cleaned count matrix
counts = pd.read_csv("/workspaces/Malaria-host-response-systems/data/processed/count_matrix_integer.txt", sep='\t', index_col=0)

# Load metadata
metadata = pd.read_csv("/workspaces/Malaria-host-response-systems/data/processed/sample_metadata_aligned.csv", index_col=0)

# Ensure count columns match metadata index order
counts = counts[metadata.index]

# Convert group to categorical with reference level 'Healthy'
metadata['group'] = pd.Categorical(metadata['group'], categories=['Healthy', 'Asymptomatic', 'Symptomatic'])
print(counts.shape)
print(metadata.head())

Compute CPM and Filter Low-Expressed Genes

In [ ]:
# Compute library sizes
lib_sizes = counts.sum(axis=0)

# Compute CPM (counts per million)
cpm = counts.div(lib_sizes, axis=1) * 1e6

# Keep genes with CPM >= 0.5 in at least 5 samples
keep_genes = (cpm >= 0.5).sum(axis=1) >= 5
counts_filtered = counts[keep_genes]

print(f"Number of genes before filtering: {counts.shape[0]}")
print(f"Number of genes after CPM filter: {counts_filtered.shape[0]}")

Create DESeqDataSet

In [ ]:
# Transpose filtered counts to samples × genes
counts_T = counts_filtered.T

# Create DESeqDataSet with filtered counts
dds = DeseqDataSet(
    counts=counts_T,
    metadata=metadata,
    design="~ group",
)

The design formula ~ group tells DESeq2 to model gene expression as a function of the group variable. The reference level is Healthy (we set it first in the categorical order). This means that by default, coefficients will compare each group to Healthy.

Run DESeq2 Pipeline

In [ ]:
# Run the full DESeq2 pipeline: normalization, dispersion estimation, and Wald test
dds.deseq2()

Verify the DESeq2 object dimensions

In [ ]:
print("Number of genes in dds:", dds.n_vars)
print("Number of samples in dds:", dds.n_obs)

Extract Results for Each Comparison
We will extract results for the three pairwise comparisons. PyDESeq2 provides a DeseqStats class for this.

Comparison 1: Symptomatic vs Healthy

In [ ]:
stat_res_sm_vs_hc = DeseqStats(dds, contrast=["group", "Symptomatic", "Healthy"])
stat_res_sm_vs_hc.summary()
res_sm_vs_hc = stat_res_sm_vs_hc.results_df
print(res_sm_vs_hc.head())

Explanation: The contrast argument specifies the comparison: ["group", "Symptomatic", "Healthy"] means coefficient for Symptomatic minus Healthy, i.e., log2 fold change is Symptomatic/Healthy.

Comparison 2: Asymptomatic vs Healthy

In [ ]:
stat_res_am_vs_hc = DeseqStats(dds, contrast=["group", "Asymptomatic", "Healthy"])
stat_res_am_vs_hc.summary()
res_am_vs_hc = stat_res_am_vs_hc.results_df

Comparison 3: Symptomatic vs Asymptomatic (Key)

In [ ]:
stat_res_sm_vs_am = DeseqStats(dds, contrast=["group", "Symptomatic", "Asymptomatic"])
stat_res_sm_vs_am.summary()
res_sm_vs_am = stat_res_sm_vs_am.results_df

Save Results

In [ ]:
res_sm_vs_hc.to_csv("/workspaces/Malaria-host-response-systems/data/processed/DESeq2_results_symptomatic_vs_healthy.csv")
res_am_vs_hc.to_csv("/workspaces/Malaria-host-response-systems/data/processed/DESeq2_results_asymptomatic_vs_healthy.csv")
res_sm_vs_am.to_csv("/workspaces/Malaria-host-response-systems/data/processed/DESeq2_results_symptomatic_vs_asymptomatic.csv")

MA Plots and Volcano Plots
We will create plots for each comparison.

MA Plot for Symptomatic vs Asymptomatic

In [ ]:
def plot_ma(res, title):
    # MA plot: x = mean expression, y = log2 fold change
    res = res.dropna(subset=['log2FoldChange', 'baseMean'])
    plt.figure(figsize=(8,6))
    plt.scatter(np.log10(res['baseMean']), res['log2FoldChange'], s=2, alpha=0.4)
    plt.axhline(y=0, color='grey', linestyle='--')
    plt.xlabel('log10(baseMean)')
    plt.ylabel('log2 fold change')
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_ma(res_sm_vs_am, 'Symptomatic vs Asymptomatic')

Volcano Plot for Symptomatic vs Asymptomatic

In [ ]:
def plot_volcano(res, title, fdr_cutoff=0.1):
    res = res.dropna(subset=['padj', 'log2FoldChange'])
    res['-log10padj'] = -np.log10(res['padj'])
    res['significant'] = res['padj'] < fdr_cutoff
    plt.figure(figsize=(8,6))
    plt.scatter(res['log2FoldChange'], res['-log10padj'], s=3, alpha=0.4, c=res['significant'], cmap='coolwarm')
    plt.axhline(y=-np.log10(fdr_cutoff), color='grey', linestyle='--')
    plt.xlabel('log2 fold change')
    plt.ylabel('-log10 adjusted p-value')
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_volcano(res_sm_vs_am, 'Symptomatic vs Asymptomatic', fdr_cutoff=0.1)

Summarize Significant Genes

In [ ]:
# Count significant genes at FDR 0.1 for each comparison
for name, res in [('Symptomatic vs Healthy', res_sm_vs_hc),
                  ('Asymptomatic vs Healthy', res_am_vs_hc),
                  ('Symptomatic vs Asymptomatic', res_sm_vs_am)]:
    n_sig = (res['padj'] < 0.1).sum()
    print(f"{name}: {n_sig} significant genes (FDR<0.1)")

Inspect the significant genes
Check the baseMean distribution of the significant genes. Many may have very low expression levels.

In [ ]:
# For the key comparison
sig_genes = res_sm_vs_am[res_sm_vs_am['padj'] < 0.1]
print(sig_genes[['baseMean', 'log2FoldChange', 'padj']].describe())